In [23]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML


c= 1.0
epsi0= 1.0
wc_wp= 0.3
ne= 200000
L = 2*np.pi       
ng= 256
dx= L/ng
dt= 0.5*dx/c
t_tot= 200*2*np.pi

wp= 1.0
wc= wc_wp  
B0z= wc                    
q_over_m = -1.0                   
q_mp= -(L/ne)                
E0= 1e-8

In [24]:
x_e = np.linspace(0, L, ne, endpoint=False)
k1  = 2*np.pi/L
Ex  = np.zeros(ng)
Ey  = E0 * np.sin(k1 * np.arange(ng) * dx)
Bz  = np.full(ng, B0z) + (E0/c) * np.sin(k1 * np.arange(ng) * dx)
Jx  = np.zeros(ng)
Jy  = np.zeros(ng)

In [25]:
Ex_his=[]
Ey_his=[]
vx_his=[]
vy_his=[]
Jx_his=[]
Jy_his=[]
Bz_his=[]


In [26]:
vx=np.zeros(ne)
vy=np.zeros(ne)

In [27]:
def getJx(vx,x_e):
    j_e= (x_e // dx).astype(int) % ng
    w_right_e = (x_e - j_e * dx) / dx
    w_left_e= 1.0 - w_right_e

    Jx=np.bincount(j_e,weights=w_left_e  * q_mp * vx, minlength=ng)
    Jx+=np.bincount((j_e+1) % ng,  weights=w_right_e * q_mp * vx,  minlength=ng)
    return Jx/dx

In [28]:
def getJy(vy,x_e):
    j_e= (x_e // dx).astype(int) % ng
    w_right_e = (x_e - j_e * dx) / dx
    w_left_e= 1.0 - w_right_e

    Jy=np.bincount(j_e,weights=w_left_e  * q_mp * vy, minlength=ng)
    Jy+=np.bincount((j_e+1) % ng,  weights=w_right_e * q_mp * vy,  minlength=ng)
    return Jy/dx

In [29]:
def getEx(J_curr,Ex):
    Ex-=J_curr*(dt)
    return Ex

In [30]:
def getBz(Ey, Bz):
    Bz -= (np.roll(Ey, -1) - Ey) / dx * (dt/2)
    return Bz

In [31]:
def getEy(Ey, Bz, Jy):
    Ey += (Bz - np.roll(Bz, 1)) / dx * dt - Jy * dt
    return Ey

In [32]:
def field_par(x_e_curr, F):
    j = (x_e_curr // dx).astype(int) % ng
    w_right = (x_e_curr - j*dx) / dx
    w_left  = 1.0 - w_right
    return w_left*F[j] + w_right*F[(j+1)%ng]

In [33]:
t = 0
while t < t_tot:
    print(f"\Entering at t = {t} s" , end="", flush=True)    # 1. Half-step B
    Bz = getBz(Ey, Bz)
    
    # 2. Deposit currents from v at t-dt/2
    Jx = getJx(vx, x_e)
    Jy = getJy(vy, x_e)
    
    # 3. Update E fields
    Ex = getEx(Jx, Ex)
    Ey = getEy(Ey, Bz, Jy)
    
    # 4. Half-step B again
    Bz = getBz(Ey, Bz)
    
    # 5. Boris push particles (v at t-dt/2 → t+dt/2)
    vx_minus = vx + q_over_m * field_par(x_e, Ex) * (dt/2)
    vy_minus = vy + q_over_m * field_par(x_e, Ey) * (dt/2)

    Bz_e  = field_par(x_e, Bz)
    t_rot = q_over_m * Bz_e * (dt/2)
    s_rot = 2*t_rot / (1 + t_rot**2)

    vx_prime = vx_minus + vy_minus * t_rot
    vy_prime = vy_minus - vx_minus * t_rot

    vx_plus = vx_minus + vy_prime * s_rot
    vy_plus = vy_minus - vx_prime * s_rot

    vx = vx_plus + q_over_m * field_par(x_e, Ex) * (dt/2)
    vy = vy_plus + q_over_m * field_par(x_e, Ey) * (dt/2)
    
    # 6. Push positions
    x_e = (x_e + vx * dt) % L
    
    # 7. Record history
    Ex_his.append(Ex.copy())
    Ey_his.append(Ey.copy())
    Bz_his.append(Bz.copy())
    vx_his.append(vx.copy())
    vy_his.append(vy.copy())
    Jx_his.append(Jx.copy())
    Jy_his.append(Jy.copy())
    
    t += dt

\Entering at t = 0 s\Entering at t = 0.01227184630308513 s\Entering at t = 0.02454369260617026 s\Entering at t = 0.03681553890925539 s\Entering at t = 0.04908738521234052 s\Entering at t = 0.06135923151542565 s\Entering at t = 0.07363107781851078 s\Entering at t = 0.0859029241215959 s\Entering at t = 0.09817477042468103 s\Entering at t = 0.11044661672776616 s\Entering at t = 0.1227184630308513 s\Entering at t = 0.1349903093339364 s\Entering at t = 0.14726215563702155 s\Entering at t = 0.1595340019401067 s\Entering at t = 0.17180584824319184 s\Entering at t = 0.18407769454627698 s\Entering at t = 0.19634954084936213 s\Entering at t = 0.20862138715244727 s\Entering at t = 0.2208932334555324 s\Entering at t = 0.23316507975861755 s\Entering at t = 0.2454369260617027 s\Entering at t = 0.25770877236478784 s\Entering at t = 0.269980618667873 s\Entering at t = 0.28225246497095813 s\Entering at t = 0.29452431127404327 s\Entering at t = 0.3067961575771284 s\Entering at t = 0.31906800388021356 s\

/tmp/ipykernel_127244/2448947818.py:23: RuntimeWarning: overflow encountered in square
  s_rot = 2*t_rot / (1 + t_rot**2)
/tmp/ipykernel_127244/2448947818.py:25: RuntimeWarning: overflow encountered in multiply
  vx_prime = vx_minus + vy_minus * t_rot
/tmp/ipykernel_127244/2448947818.py:29: RuntimeWarning: invalid value encountered in multiply
  vy_plus = vy_minus - vx_prime * s_rot
/tmp/ipykernel_127244/1327615273.py:2: RuntimeWarning: invalid value encountered in cast
  j_e= (x_e // dx).astype(int) % ng


\Entering at t = 5.12963175468961 s\Entering at t = 5.141903600992696 s\Entering at t = 5.154175447295781 s\Entering at t = 5.166447293598867 s\Entering at t = 5.178719139901952 s\Entering at t = 5.190990986205038 s\Entering at t = 5.203262832508123 s\Entering at t = 5.215534678811209 s\Entering at t = 5.2278065251142944 s\Entering at t = 5.24007837141738 s\Entering at t = 5.2523502177204655 s\Entering at t = 5.264622064023551 s\Entering at t = 5.276893910326637 s\Entering at t = 5.289165756629722 s\Entering at t = 5.301437602932808 s\Entering at t = 5.313709449235893 s\Entering at t = 5.325981295538979 s\Entering at t = 5.338253141842064 s\Entering at t = 5.35052498814515 s\Entering at t = 5.362796834448235 s\Entering at t = 5.375068680751321 s\Entering at t = 5.387340527054406 s\Entering at t = 5.399612373357492 s\Entering at t = 5.411884219660577 s\Entering at t = 5.424156065963663 s\Entering at t = 5.4364279122667485 s\Entering at t = 5.448699758569834 s\Entering at t = 5.460971604

KeyboardInterrupt: 

In [ ]:
t=0
while t<t_tot:
    Ex_his.append(Ex.copy())
    vx_his.append(vx.copy())
    Ey_his.append(Ey.copy())
    Bz_his.append(Bz.copy())
    vy_his.append(vy.copy())
    Jx_his.append(Jx.copy())
    Jy_his.append(Jy.copy())
    
    Bz=getBz(Ey,Bz) 
    Jx=getJx(vx,x_e) 
    Ex=getEx(Jx,Ex) 

    Jy=getJy(vy,x_e) 
    Ey=getEy(Ey,Bz,Jy) 
    Bz = getBz(Ey, Bz)
    
    vx_minus = vx + (q_over_m)*field_par(x_e,Ex)*(dt/2)
    vy_minus = vy + (q_over_m)*field_par(x_e,Ey)*(dt/2)

    # Magnetic rotation (Boris)
    Bz_e   = field_par(x_e, Bz)
    t_rot  = (q_over_m)*Bz_e*(dt/2)
    s_rot  = 2*t_rot/(1 + t_rot**2)

    vx_prime = vx_minus + vy_minus*t_rot
    vy_prime = vy_minus - vx_minus*t_rot

    vx_plus  = vx_minus + vy_prime*s_rot
    vy_plus  = vy_minus - vx_prime*s_rot

    # Half electric acceleration
    vx = vx_plus + (q_over_m)*field_par(x_e,Ex)*(dt/2)
    vy = vy_plus + (q_over_m)*field_par(x_e,Ey)*(dt/2)
    x_e=(x_e+vx*dt)%L 
    t=t+dt